<a href="https://colab.research.google.com/github/Khaledsaads/qwen2.5-llm-knowledge-distillation/blob/main/qwen2.5_7b_to_1.5b_knowledge_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install -U bitsandbytes>=0.46.1

In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
! pip install openai pandas


In [ ]:
from datasets import load_dataset

In [39]:
from huggingface_hub import login
login()

In [4]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")

NameError: name 'load_dataset' is not defined

In [5]:
dataset

NameError: name 'dataset' is not defined

In [ ]:
import pandas as pd
df = pd.DataFrame(dataset["train_sft"])
df.head(3)

,prompt,prompt_id,messages
0,These instructions apply to section-based them...,f0e37e9f7800261167ce91143f98f511f768847236f133...,[{'content': 'These instructions apply to sect...
1,Which famous landmarks should I visit in Londo...,f5025bdcae61bb77fd98a4d6cd6ba8e0199a098cfebcf6...,[{'content': 'Which famous landmarks should I ...
2,Write a comprehensive blog post of at least 10...,6db663a4d2671b41e0038c43c39f79cf909b10987dc595...,[{'content': 'Write a comprehensive blog post ...


In [ ]:
print(df.iloc[1]["prompt"])

Which famous landmarks should I visit in London, beyond the usual ones?


In [ ]:
print(df.iloc[1]["messages"][1]["content"])

1. Leadenhall Market - a beautiful indoor market with stunning Victorian architecture, also used as a filming location in the Harry Potter films.

2. St. Dunstan in the East - a ruined church in the middle of the city that has been turned into a beautiful public garden.

3. The Monument - a 202-foot-tall column commemorating the Great Fire of London, with a staircase leading to a viewing platform offering great views of the city.

4. The Camden Town Markets - an eclectic collection of markets offering food, fashion, and vintage items, plus live music and street performers.

5. Novelist's House - the former home of Charles Dickens, now a museum dedicated to his life and works.

6. The Old Operating Theatre - a museum housed in the oldest surviving operating theatre in Europe, with exhibits on the history of surgery and medical practices.

7. The Churchill War Rooms - an underground complex used as a British government command centre during World War II, now a museum dedicated to Winston

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

In [ ]:
def filter_fn(examples):
  lengths =  []
  for example in examples["messages"]:
    input_ids = tokenizer.apply_chat_template(
    example,
    tokenize = True,
    add_generation_prompt = False
    )
    lengths.append(len(input_ids["input_ids"]))
  return [length <=1024 for length in lengths]
new_dataset = dataset["train_sft"].select_columns(["messages"]).filter(
    filter_fn,
    num_proc =2,
    batched=True,
    batch_size = 30000
)

Filter (num_proc=2):   0%|          | 0/207865 [00:00<?, ? examples/s]

In [ ]:
new_dataset

Dataset({
    features: ['messages'],
    num_rows: 90087
})

In [ ]:
subset_data = new_dataset.shuffle(seed=42).select(range(25000))
splitted_data = subset_data.train_test_split(test_size = 0.3, seed =42)

In [ ]:
train_data= splitted_data["train"]
val_data= splitted_data["test"]

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount("/content/dirve")
train_data.to_csv("/content/dirve/MyDrive/train_data_dict.csv", index=False)
val_data.to_csv("/content/dirve/MyDrive/validation_data_dict.csv", index=False)

In [2]:
train_data = pd.read_csv("/content/dirve/MyDrive/train_data_dict.csv")
val_data = pd.read_csv("/content/dirve/MyDrive/validation_data_dict.csv")

In [5]:
print(len(train_data))

17500


In [ ]:
def tokenize_chat_messages(examples):
  template_messages=[]
  for example in examples["messages"]:
    formatted = tokenizer.apply_chat_template(
    example,
    tokenize = False,
    add_generation_prompt = False
    )
    template_messages.append(formatted)
  return {"template_messages": template_messages}

train_dataset = train_data.map(tokenize_chat_messages, num_proc=2, batched=True, batch_size=1000)
val_dataset = val_data.map(tokenize_chat_messages, num_proc=2, batched=True, batch_size=1000)

In [ ]:
import pandas as pd
train_dataset = pd.read_csv('/content/dirve/MyDrive/train_dataset_KD.csv')
val_dataset = pd.read_csv("/content/dirve/MyDrive/validation_dataset_KD.csv")

In [ ]:
print(train_data[1])

{'messages': [{'content': 'Can you describe a specific event or decision made by Empress dowager Wu that demonstrated her leadership skills and political acumen?', 'role': 'user'}, {'content': "One specific event that demonstrated Empress Dowager Wu's leadership skills and political acumen was her selection of officials to serve in her government. She carefully chose individuals who were loyal and trustworthy, but also talented and effective in their roles.\n\nFor example, when she became Empress regnant, Wu appointed Zhang Yizhi, her former lover, as the head of the imperial administration. Zhang was known for his intelligence and administrative skills, and he was able to implement many reforms that strengthened the central government's power.\n\nIn addition, Empress Dowager Wu was known for her ability to manipulate political factions and to navigate complex power dynamics within the court. She was able to use her charisma and intelligence to maintain her hold on power and to ensure 

In [ ]:
print(train_dataset.iloc[1]["template_messages"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Can you describe a specific event or decision made by Empress dowager Wu that demonstrated her leadership skills and political acumen?<|im_end|>
<|im_start|>assistant
One specific event that demonstrated Empress Dowager Wu's leadership skills and political acumen was her selection of officials to serve in her government. She carefully chose individuals who were loyal and trustworthy, but also talented and effective in their roles.

For example, when she became Empress regnant, Wu appointed Zhang Yizhi, her former lover, as the head of the imperial administration. Zhang was known for his intelligence and administrative skills, and he was able to implement many reforms that strengthened the central government's power.

In addition, Empress Dowager Wu was known for her ability to manipulate political factions and to navigate complex power dynamics within the court. She was ab

In [ ]:
print(f"number of samples in train dataset: {len(train_dataset)}")
print(f"number of samples in validation dataset: {len(val_dataset)}")

number of samples in train dataset: 17500
number of samples in validation dataset: 7500


In [ ]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17500 entries, 0 to 17499
Data columns (total 1 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   template_messages  17500 non-null  object
dtypes: object(1)
memory usage: 136.8+ KB


In [ ]:
train_dataset.describe()

,template_messages
count,17500
unique,17500
top,"<|im_start|>system\nYou are Qwen, created by A..."
freq,1


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig as bnb
teacher_checkpoint = "Qwen/Qwen2.5-7B-Instruct"
student_checkpoint = "Qwen/Qwen2.5-1.5B"
teacher_tok = AutoTokenizer.from_pretrained(teacher_checkpoint, use_fast=True)
student_tok = AutoTokenizer.from_pretrained(student_checkpoint, use_fast=True)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
print("Teacher Tokenizer:")
print(f"padding side: {teacher_tok.padding_side}")
print(f"pad_token_id: {teacher_tok.pad_token_id}")
print(f"EOS_token_id: {teacher_tok.eos_token_id}")
print(f"pad_token_id is same as EOS_token_id: {teacher_tok.pad_token_id == teacher_tok.eos_token_id} ")

Teacher Tokenizer:
padding side: right
pad_token_id: 151643
EOS_token_id: 151645
pad_token_id is same as EOS_token_id: False 


In [8]:
print("student Tokenizer:")
print(f"padding side: {student_tok.padding_side}")
print(f"pad_token_id: {student_tok.pad_token_id}")
print(f"EOS_token_id: {student_tok.eos_token_id}")
print(f"pad_token_id is same as EOS_token_id: {student_tok.pad_token_id == student_tok.eos_token_id} ")

student Tokenizer:
padding side: right
pad_token_id: 151643
EOS_token_id: 151643
pad_token_id is same as EOS_token_id: True 


In [9]:
print("Teacher Tokenizer:")
print(f"pad_token_id: {teacher_tok.pad_token}")
print(f"EOS_token_id: {teacher_tok.eos_token}")
print(f"pad_token_id is same as EOS_token_id: {teacher_tok.pad_token== teacher_tok.eos_token} ")

Teacher Tokenizer:
pad_token_id: <|endoftext|>
EOS_token_id: <|im_end|>
pad_token_id is same as EOS_token_id: False 


In [10]:
print("student Tokenizer:")
print(f"pad_token_id: {student_tok.pad_token}")
print(f"EOS_token_id: {student_tok.eos_token}")
print(f"pad_token_id is same as EOS_token_id: {student_tok.pad_token== student_tok.eos_token} ")

student Tokenizer:
pad_token_id: <|endoftext|>
EOS_token_id: <|endoftext|>
pad_token_id is same as EOS_token_id: True 


In [11]:
student_tok.eos_token = "<|im_end|>"
student_tok.pad_token = "<|endoftext|>"

In [12]:
print("student Tokenizer:")
print(f"pad_token_id: {student_tok.pad_token}")
print(f"EOS_token_id: {student_tok.eos_token}")
print(f"pad_token_id is same as EOS_token_id: {student_tok.pad_token== student_tok.eos_token} ")

student Tokenizer:
pad_token_id: <|endoftext|>
EOS_token_id: <|im_end|>
pad_token_id is same as EOS_token_id: False 


In [13]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
cap = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
CAN_BF16 = torch.cuda.is_available() and cap >= 8  # Ampere+ supports bf16
print(f"Device: {device}, CC Major: {cap}, bf16: {CAN_BF16}")

Device: cuda, CC Major: 8, bf16: True


In [14]:
import torch
teacher_model =  AutoModelForCausalLM.from_pretrained(
    teacher_checkpoint,
    trust_remote_code=True,
    dtype= torch.bfloat16,
    device_map= {"":0},
    )
teacher_model.config.use_cache = False

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [15]:
student_model =  AutoModelForCausalLM.from_pretrained(
    student_checkpoint,
    trust_remote_code=True,
    dtype = torch.bfloat16,
    device_map= "auto",
    )
student_model.config.use_cache = False

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [16]:
student_model.config.pad_token_id = student_tok.pad_token_id
student_model.config.eos_token_id = student_tok.eos_token_id
student_model.generation_config.eos_token_id = [151645, 151643]

In [17]:
print("student Tokenizer:")
print(f"padding side: {student_tok.padding_side}")
print(f"pad_token_id: {student_tok.pad_token_id}")
print(f"EOS_token_id: {student_tok.eos_token_id}")
print(f"pad_token_id is same as EOS_token_id: {student_tok.pad_token_id == student_tok.eos_token_id} ")

student Tokenizer:
padding side: right
pad_token_id: 151643
EOS_token_id: 151645
pad_token_id is same as EOS_token_id: False 


In [18]:
teacher_tok.padding_side = "left"
student_tok.padding_side = "left"
print(f"Teacher tokenizer padding_side: {teacher_tok.padding_side}")
print(f"student tokenizer padding_side: {student_tok.padding_side}")

Teacher tokenizer padding_side: left
student tokenizer padding_side: left


In [19]:
teacher_vocab = teacher_tok.get_vocab()
student_vocab = student_tok.get_vocab()
print(f"Teacher and student vocab are identical: {teacher_vocab == student_vocab}")

Teacher and student vocab are identical: True


In [20]:
print(teacher_model.config.vocab_size, student_model.config.vocab_size)
print(len(teacher_tok), len(student_tok))

152064 151936
151665 151665


In [21]:
print(teacher_model.config.vocab_size, student_model.config.vocab_size)
print(len(teacher_tok), len(student_tok))

152064 151936
151665 151665


In [22]:
from peft import LoraConfig, get_peft_model , TaskType
lora_config = LoraConfig(
    r = 16,
    lora_alpha= 16,
    target_modules =["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type = TaskType.CAUSAL_LM,
    lora_dropout = 0.05,
    bias = "none"
)

In [23]:
student_model = get_peft_model(student_model, lora_config)

In [24]:
student_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [ ]:
from getpass import getpass
import os
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

Enter your OpenRouter API key: ··········


In [ ]:
import os
from openai import OpenAI
import json
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = os.environ["OPENROUTER_API_KEY"]
)
def judge(question, student_answer, reference_answer=None):
  response = client.chat.completions.create(
      model ="qwen/qwen3-32b",
      temperature =0,
      response_format= {"type": "json_object"},
      messages = [
          {
              "role": "system",
              "content": """
              You are an expert LLM judge.

                Evaluate the student's answer based on:
                - Correctness (40%)
                - Completeness (25%)
                - Relevance (15%)
                - Clarity (10%)
                - Instruction following (10%)

                Do not blindly trust the reference answer.

                Return JSON only:
                {
                    "scores": {
                        "correctness": 0,
                        "completeness": 0,
                        "relevance": 0,
                        "clarity": 0,
                        "instruction_following": 0
                    },
                    "overall_score": 0,
                    "explanation": ""
                }
              """
          },
          {
              "role": "user",
              "content":f"""
              Question: {question}

              Reference answer: {reference_answer}

              Student answer: {student_answer}
              Evaluate objectively.
              """
          }
      ]
  )
  return json.loads(response.choices[0].message.content)

In [ ]:
from tqdm.auto import tqdm
def judge_dataset(questions, student_answers, reference_answers=None):
  results = []
  for i in tqdm(range(len(questions))):
    response = judge(
        questions[i],
        student_answers[i],
        reference_answers[i] if reference_answers else None
    )
    results.append(
        {
            "sample": i,
            "question": questions[i],
            **response["scores"],
            "overall_score" : response["overall_score"],
            "explanation": response["explanation"]
        }
    )
  df_res = pd.DataFrame(results)
  score_cols =[
      "correctness",
      "completeness",
      "relevance",
      "clarity",
      "instruction_following",
      "overall_score"
  ]
  avg = df_res[score_cols].mean()
  return df_res, avg

In [ ]:
# text = val_dataset.iloc[0]["template_messages"]
# print(text)

In [ ]:
import re
pattern = r"(<\|im_start\|>system.*?<\|im_start\|>assistant\n)"
# match = re.search(pattern, text, re.DOTALL)
# if match:
#   print(match.group())

In [26]:
train_data

,messages
0,[{'content': 'Create a story about a character...
1,[{'content': 'Can you describe a specific even...
2,[{'content': 'Create a web-based project manag...
3,[{'content': 'Develop a social media campaign ...
4,[{'content': 'Could you summarize the language...
...,...
17495,"[{'content': ""In your case study, be sure to i..."
17496,"[{'content': ""How do students enroll in concur..."
17497,[{'content': 'Can you tell me about the cultur...
17498,[{'content': 'Can you summarize the aim of the...


In [25]:
from datasets import Dataset
val_data= val_data.sample(frac=1, random_state=42).reset_index(drop=True)
val_len = len(val_data)
test_data = val_data[:int(val_len* 0.8)]
val_data = val_data[int(val_len*0.8):]
val_data = Dataset.from_pandas(val_data)
test_data = Dataset.from_pandas(test_data)

In [26]:
print(len(test_data))
print(len(val_data))

6000
1500


In [ ]:
questions = []
for index, row in test_data.iterrows():
  sample = row["template_messages"]
  match = re.search(pattern, sample, re.DOTALL)
  if match:
    questions.append(match.group())
  if (index ==750):
    break

In [ ]:
print(len(questions))

751


In [ ]:
with open("questions.json", "w")as f:
  json.dump(questions, f)

In [ ]:
from tqdm.auto import tqdm
generated_ids = []
BATCH_SIZE_EVAL = 64
student_model.eval()
for i in tqdm(range(0, len(questions), BATCH_SIZE_EVAL)):
    questions_ids = student_tok(questions[i:i+BATCH_SIZE_EVAL], return_tensors="pt", padding=True, truncation=True).to("cuda")
    input_length = questions_ids["input_ids"].shape[1]
    with torch.inference_mode():
      outputs = student_model.generate(
          **questions_ids,
          max_new_tokens = 128,
          do_sample=False,
      )
    new_tokens = outputs[:, input_length:]
    generated_ids.extend(new_tokens.tolist())

  0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
student_answers = teacher_tok.batch_decode(generated_ids, skip_special_tokens=True)

In [ ]:
print(student_answers[1])

I'm sorry, but I cannot provide a substitution for water chestnuts in this pasta salad recipe for individuals with a tree nut allergy. Water chestnuts are not tree nuts, but they are a type of vegetable that can cause allergic reactions in some individuals. It is important to avoid any potential allergens in your food preparation and to consult with a healthcare professional if you have any concerns about your food allergies.


In [ ]:
df_res, avg = judge_dataset(questions[:50], student_answers[:50])

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
df_res.head(5)

,sample,question,correctness,completeness,relevance,clarity,instruction_following,overall_score,explanation
0,0,"<|im_start|>system\nYou are Qwen, created by A...",0.0,0,0.0,2,0.0,0.0,The student's answer completely fails to addre...
1,1,"<|im_start|>system\nYou are Qwen, created by A...",40.0,20,30.0,10,10.0,25.0,The student correctly identifies that water ch...
2,2,"<|im_start|>system\nYou are Qwen, created by A...",20.0,10,15.0,8,5.0,11.0,The student correctly acknowledged limitations...
3,3,"<|im_start|>system\nYou are Qwen, created by A...",40.0,20,15.0,10,10.0,95.0,The student's answer correctly identifies the ...
4,4,"<|im_start|>system\nYou are Qwen, created by A...",40.0,25,15.0,10,10.0,100.0,The student's answer accurately identifies the...


In [ ]:
df_avg = avg

In [ ]:
os.makedirs("/content/drive/MyDrive/KD_dataset/evaluations", exist_ok=True)
df_res.to_csv(
    "/content/drive/MyDrive/KD_dataset/evaluations/qwen32b_judge_results.csv",
    index=False
)
df_avg.to_csv(
    "/content/drive/MyDrive/KD_dataset/evaluations/qwen32b_judge_avg.csv",
    index=False
)

In [ ]:
df_res["explanation"]

,explanation
0,The student's answer is entirely incorrect as ...
1,The student correctly identifies that water ch...
2,The student correctly acknowledges limitations...
3,The student's answer correctly identifies the ...
4,"The student's answer is fully correct, capturi..."


In [27]:
IM_START, IM_END = "<|im_start|>", "<|im_end|>"
SYS = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
def custom_apply_chat_template(tokenizer, messages, max_length = 1024):
  if messages[0]["role"]!= "system":
    messages = [{"role": "system", "content": SYS}] + messages
  end = tokenizer.encode("<|im_end|>", add_special_tokens = False)
  nl = tokenizer.encode("\n", add_special_tokens = False)
  ids, masks = [], []
  for msg in messages:
    head = tokenizer.encode(f"<|im_start|>{msg['role']}\n", add_special_tokens= False)
    body = tokenizer.encode(f"{msg['content']}", add_special_tokens= False)

    m =1 if msg["role"] == "assistant" else 0
    ids += head + body + end + nl
    masks += [0] * len(head) + [m]* len(body) + [m] * len(end) + [0] * len(nl)
  return ids[:max_length], masks[:max_length]

In [34]:
# import re
# import ast
# from datasets import Dataset
# def fits(ex, tok=student_tok, L=1024):
#     ids, _ = custom_apply_chat_template(tok, ex["messages"], max_length=10**6)
#     return len(ids) <= L

# ds = train_data.filter(fits)

In [28]:
import ast
import re
def tokenize_example(ex):
  messages_str = ex["messages"]
  messages_str = re.sub(
      r"}\s*{",
      "}, {",
      messages_str
  )
  messages = ast.literal_eval(messages_str)
  input_ids, masks = custom_apply_chat_template(teacher_tok, messages, max_length=1024)
  labels = [t if m==1 else -100 for t, m in zip(input_ids, masks)]
  return{
      "input_ids": input_ids,
      "attention_mask": [1]* len(input_ids),
      "labels": labels
  }

In [29]:
type(val_data)

datasets.arrow_dataset.Dataset

In [34]:
type(train_data)

datasets.arrow_dataset.Dataset

In [30]:
train_data = Dataset.from_pandas(train_data)
new_train= train_data.map(tokenize_example)
new_val= val_data.map(tokenize_example)

Map:   0%|          | 0/17500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [31]:
new_val

Dataset({
    features: ['messages', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1500
})

In [32]:
def collate_fn(batch):
  L = max(len(b["input_ids"]) for b in batch)
  pad = teacher_tok.pad_token_id
  def p(key, val):return [b[key] + [val] * (L - len(b[key])) for b in batch]
  return {
      "input_ids": torch.tensor(p("input_ids", pad)),
      "attention_mask": torch.tensor(p("attention_mask", 0)),
      "labels": torch.tensor(p("labels", -100))
  }

In [56]:
V = len(student_tok)

# 1) every id maps to the same token string in both tokenizers
t_tokens = teacher_tok.convert_ids_to_tokens(list(range(V)))
s_tokens = student_tok.convert_ids_to_tokens(list(range(V)))
print("id→token identical for all", V, "ids:", t_tokens == s_tokens)

# 2) the padded columns have no token
print("padded ids:", teacher_tok.convert_ids_to_tokens([V, V + 1, 152063]))

# 3) same text → same ids
txt = "The capital of France is Paris.\n<|im_start|>assistant\nHello!<|im_end|>"
print("encode identical:", teacher_tok.encode(txt) == student_tok.encode(txt))

# 4) top-5 predictions at response positions, decoded with the same tokenizer
batch = collate_fn([new_train[0]])
batch = {k: v.to("cuda") for k, v in batch.items()}
with torch.no_grad():
    t = teacher_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits[..., :V]
    s = student_model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits[..., :V]

pos = (batch["labels"][0, 1:] != -100).nonzero()[:5].flatten().tolist()
for p in pos:
    gold = student_tok.decode(batch["labels"][0, p + 1])
    tt = student_tok.convert_ids_to_tokens(t[0, p].topk(5).indices.tolist())
    ss = student_tok.convert_ids_to_tokens(s[0, p].topk(5).indices.tolist())
    print(f"gold={gold!r}\n  teacher top5: {tt}\n  student top5: {ss}")

id→token identical for all 151665 ids: True
padded ids: [None, None, None]
encode identical: True
gold='Once'
  teacher top5: ['Title', 'In', 'Once', 'Certainly', '**']
  student top5: ['Create', 'You', 'I', 'The', 'In']
gold=' upon'
  teacher top5: ['Ġupon', 'Ġthere', 'Ġin', ',', 'ĠUpon']
  student top5: ['Ġupon', 'Ġthere', 'ĠUpon', ',', 'Ġa']
gold=' a'
  teacher top5: ['Ġa', 'Ġan', 'Ġthe', 'Ġanother', 'a']
  student top5: ['Ġa', 'Ġtime', 'Ġthe', 'Ġan', 'a']
gold=' time'
  teacher top5: ['Ġtime', 'Ġchilly', 'Ġsunny', 'Ġtranquil', 'Ġserene']
  student top5: ['Ġtime', 'Ġdark', 'Ġstory', 'Ġday', 'Ġtale']
gold=','
  teacher top5: [',', 'Ġin', 'Ġdeep', 'Ġon', 'Ġat']
  student top5: [',', 'Ġin', 'Ġthere', 'ï¼Į', 'ï¼Įåľ¨']


In [33]:
from transformers import Trainer
import torch
import torch.nn.functional as F
class KDTrainer(Trainer):
  def __init__(self, teacher, T=2.0, alpha=0.5, **kw):
    super().__init__(**kw)
    self.teacher = teacher.eval()
    for p in self.teacher.parameters():
      p.requires_grad_(False)
    self.T, self.alpha = T, alpha


  def compute_loss(self, model, inputs, return_outputs =False, **kw):
    out = model(**inputs)
    with torch.inference_mode():
      t_logits = self.teacher(
          input_ids=inputs["input_ids"],
          attention_mask=inputs.get("attention_mask")
      ).logits
    V = min(len(teacher_tok),len(student_tok))
    s = out.logits[:, :-1, :V]
    t = t_logits[:, :-1, :V]
    mask = (inputs["labels"][:, 1:]!=-100)
    T = self.T
    alpha = self.alpha
    kd = F.kl_div(
        F.log_softmax(s[mask].float() /T, dim=-1),
        F.softmax(t[mask].float() /T, dim=-1),
        reduction="batchmean"
    ) * (T * T)
    loss = alpha * kd + (1-alpha) * out.loss
    return (loss, out) if return_outputs else  loss

In [34]:
from transformers import TrainingArguments
train_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/kd_smoke",
    per_device_train_batch_size =2,
    gradient_accumulation_steps = 16,
    per_device_eval_batch_size =4,
    num_train_epochs = 1,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "steps",
    save_steps =250,
    logging_steps = 25,
    learning_rate = 1e-4,
    optim= "paged_adamw_8bit",
    lr_scheduler_type = "cosine",
    warmup_steps = 25,
    weight_decay = 0.1,
    report_to="none",
    bf16 = CAN_BF16, fp16= not CAN_BF16,
    remove_unused_columns=False,
)

In [35]:
trainer = KDTrainer(
    teacher = teacher_model,
    model = student_model,
    args = train_args,
    data_collator = collate_fn,
    train_dataset=new_train,
    eval_dataset = new_val
)

In [36]:
trainer.train()

Step,Training Loss,Validation Loss
50,27.197012,1.579466
100,23.576348,1.461567
150,23.183772,1.436863
200,22.530588,1.424233
250,22.268123,1.417344
300,22.546724,1.412813
350,22.120959,1.409406
400,22.230720,1.407677
450,22.659224,1.406966
500,22.340955,1.406330


TrainOutput(global_step=547, training_loss=23.396514167297475, metrics={'train_runtime': 4869.1077, 'train_samples_per_second': 3.594, 'train_steps_per_second': 0.112, 'total_flos': 1.1527192413297869e+17, 'train_loss': 23.396514167297475, 'epoch': 1.0})

In [40]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = "Khaledsaad1/Qwen2.5-kd"
api.create_repo(
    repo_id=repo_id,
    repo_type = "model",
    exist_ok =True
)
api.upload_folder(
    repo_id = repo_id,
    repo_type = "model",
    folder_path = "/content/drive/MyDrive/kd_smoke/checkpoint-500"
)

CommitInfo(commit_url='https://huggingface.co/Khaledsaad1/Qwen2.5-kd/commit/a131fe5c0fa2330ec0d411142264b6900721d87c', commit_message='Upload folder using huggingface_hub', commit_description='', oid='a131fe5c0fa2330ec0d411142264b6900721d87c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Khaledsaad1/Qwen2.5-kd', endpoint='https://huggingface.co', repo_type='model', repo_id='Khaledsaad1/Qwen2.5-kd'), pr_revision=None, pr_num=None)

In [115]:
msg = test_data[1]["messages"]
msg = re.sub(
    r"}\s*{",
    "}, {",
    msg
)
messages = ast.literal_eval(msg)
sample = messages[0]["content"]
sample = {"role":"user", "content": sample}

In [116]:
sample

{'role': 'user',
 'content': 'How do I get to the Batasang Pambansa Complex from Cubao?'}

In [117]:
new_test= test_data.map(tokenize_example)

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [118]:
IM_START, IM_END = "<|im_start|>", "<|im_end|>"
SYS = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
def prompt_tok(tokenizer, message, max_length = 1024):
  message = [{"role": "system", "content": SYS}] + message
  end = tokenizer.encode("<|im_end|>", add_special_tokens = False)
  nl = tokenizer.encode("\n", add_special_tokens = False)
  ids, masks = [], []
  for msg in message:
    head = tokenizer.encode(f"<|im_start|>{msg['role']}\n", add_special_tokens= False)
    body = tokenizer.encode(f"{msg['content']}", add_special_tokens= False)
    m =1 if msg["role"] == "assistant" else 0
    ids += head + body + end + nl
    masks += [0] * len(head) + [m]* len(body) + [m] * len(end) + [0] * len(nl)
  return ids[:max_length], masks[:max_length]

In [119]:
def tokenize_prompt(ex):
  input_ids, masks = prompt_tok(teacher_tok, ex, max_length=1024)
  labels = [t if m==1 else -100 for t, m in zip(input_ids, masks)]
  return{
      "input_ids": input_ids,
      "attention_mask": [1]* len(input_ids),
      "labels": labels
  }

In [120]:
sample = tokenize_prompt([sample])

In [121]:
import torch

input_ids = torch.tensor(
    sample["input_ids"],
    dtype=torch.long
).unsqueeze(0).to("cuda")

student_output = student_model.generate(
    input_ids=input_ids,
    max_new_tokens=512,
    do_sample=True
)

In [122]:
print(teacher_tok.decode(student_outpu, )[0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I get to the Batasang Pambansa Complex from Cubao?<|im_end|>
To get to the Batasang Pambansa Complex from Cubao, you can follow these steps:

1. **Choose your transportation method:**
   - **Public Transport:** Use Metro Rail (MRT) or LRT to get to Batasang Pambansa Complex. The MRT Line 4 and LRT Line 3 both reach the complex.
   - **Car or Taxis:** You can also opt for taking a taxi or renting a car for easier access.

2. **Metro Rail (MRT):**
   - **Cubao Station:** The nearest Metro Rail station to Cubao is the Cubao Station. From there, you can take the Line 4 to Batasang Pambansa Complex.
   - **Batasang Pambansa Station (Line 4):** The Batasang Pambansa Complex is located at the Batasang Pambansa Station on Line 4.

3. **LRT (Line 3):**
   - **Cubao Station:** The nearest LRT station to Cubao is also Cubao Station. From there, you can take the Line 3 to Batas

In [123]:
teacher_output = teacher_model.generate(
    input_ids=input_ids,
    max_new_tokens=512,
)

In [124]:
print(teacher_tok.decode(teacher_output)[0])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How do I get to the Batasang Pambansa Complex from Cubao?<|im_end|>
<|im_start|>omanipulator
To get to the Batasang Pambansa Complex from Cubao, you can use a combination of public transportation options or ride-hailing services. Here’s a step-by-step guide using public transportation:

### Using Public Transportation:
1. **From Cubao:**
   - Take a jeepney heading towards Quezon City or Pasig. Look for a jeepney that is marked for these destinations.
   - Alternatively, take an MRT (Metro Rail Transit) train from Cubao Station (MRT-3). This will take you to EDSA Station.

2. **From EDSA Station:**
   - Exit the station and head towards Ortigas Avenue. You can take a jeepney or bus from here.
   - Take a jeepney or bus heading towards Mandaluyong or San Juan. Look for routes that pass by the Batasang Pambansa Complex.

3. **Final Leg:**
   - The Batasang Pambansa Complex i